In [ ]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb 'timesfm[torch]'

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

In [ ]:
import numpy as np
import pandas as pd
import torch
import mlflow
import mlflow.sklearn
import wandb
import timesfm
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
train.shape

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [ ]:
# TimesFM: google-is pretrained foundation modeli. treningi ar gvchirdeba,
# yovel seriaze zero-shot prognozs vigebt. modeli pickle-shi ar inaxeba
# (200M+ parametria), predict-is dros huggingface-dan chamoitvirteba.
# bibliotekas ori versiis api aqvs (1.x da 2.5+), orive mxardacherilia
NEW_TIMESFM_API = not hasattr(timesfm, 'TimesFm')

class TimesFMForecaster(BaseEstimator):
    _shared_model = None

    def __init__(self, context_len=128, horizon_len=48, batch_size=64):
        self.context_len = context_len
        self.horizon_len = horizon_len
        self.batch_size = batch_size

    def _model(self):
        cls = TimesFMForecaster
        if cls._shared_model is None:
            if NEW_TIMESFM_API:
                m = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
                m.compile(timesfm.ForecastConfig(
                    max_context=max(self.context_len, 256),
                    max_horizon=max(self.horizon_len, 64),
                    normalize_inputs=True,
                ))
                cls._shared_model = m
            else:
                cls._shared_model = timesfm.TimesFm(
                    hparams=timesfm.TimesFmHparams(
                        backend='gpu' if torch.cuda.is_available() else 'cpu',
                        per_core_batch_size=self.batch_size,
                        context_len=self.context_len,
                        horizon_len=self.horizon_len,
                    ),
                    checkpoint=timesfm.TimesFmCheckpoint(
                        huggingface_repo_id='google/timesfm-2.0-500m-pytorch'),
                )
        return cls._shared_model

    def _forecast(self, contexts):
        model = self._model()
        outs = []
        for i in range(0, len(contexts), self.batch_size):
            chunk = contexts[i:i + self.batch_size]
            if NEW_TIMESFM_API:
                point, _ = model.forecast(horizon=self.horizon_len, inputs=chunk)
            else:
                point, _ = model.forecast(chunk, freq=[1] * len(chunk))
            outs.append(np.asarray(point))
        return np.concatenate(outs, axis=0)

    def fit(self, X, y):
        df = X[['Store', 'Dept', 'Date']].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['Weekly_Sales'] = y.values if hasattr(y, 'values') else y

        self.history_ = df.pivot_table(index=['Store', 'Dept'], columns='Date',
                                       values='Weekly_Sales').sort_index(axis=1)
        self.series_mean_ = self.history_.mean(axis=1).fillna(0)
        self.last_date_ = self.history_.columns.max()
        return self

    def predict(self, X):
        df = X.copy().reset_index(drop=True)
        df['Date'] = pd.to_datetime(df['Date'])

        contexts = []
        for k in self.history_.index:
            v = self.history_.loc[k].dropna().values[-self.context_len:]
            contexts.append(v if len(v) else np.array([0.0]))

        point = self._forecast(contexts)

        future_dates = pd.date_range(self.last_date_ + pd.Timedelta(weeks=1),
                                     periods=self.horizon_len, freq='W-FRI')
        n = min(point.shape[1], len(future_dates))
        fc = pd.DataFrame(point[:, :n], index=self.history_.index, columns=future_dates[:n])
        fc_long = fc.stack()

        idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept'], df['Date']])
        vals = fc_long.reindex(idx).values.astype(float)
        pair_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept']])
        fallback = self.series_mean_.reindex(pair_idx).values.astype(float)
        global_mean = float(self.series_mean_.mean())

        preds = np.where(np.isnan(vals), np.where(np.isnan(fallback), global_mean, fallback), vals)
        return np.clip(preds, 0, None)

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('TimesFM_Training')

In [ ]:
with mlflow.start_run(run_name='TimesFM_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='TimesFM_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    wandb.log({'negative_sales_rows': n_negative})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

In [ ]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

# test-formis holdout: oqtombramde vswavlobt, shemdegi noembridan ivlisamde vafasebt.
# diagnostikam achvena rom es fanjara kaggle-is tests bevrad ukot imeorebs vidre CV
HOLDOUT_CUTOFF = pd.Timestamp('2011-10-28')
HOLDOUT_END = pd.Timestamp('2012-07-27')

In [ ]:
params = dict(context_len=128, horizon_len=48)

with mlflow.start_run(run_name='TimesFM_CV'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='TimesFM_CV', config=params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        tm = train['Date'] <= train_end
        vm = (train['Date'] > train_end) & (train['Date'] <= val_end)

        model = TimesFMForecaster(**params)
        model.fit(train.loc[tm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[tm])
        preds = model.predict(train.loc[vm, ['Store', 'Dept', 'Date', 'IsHoliday']])

        score = wmae(y_train[vm].values, preds, train.loc[vm, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    hm = train['Date'] <= HOLDOUT_CUTOFF
    hv = (train['Date'] > HOLDOUT_CUTOFF) & (train['Date'] <= HOLDOUT_END)
    hmodel = TimesFMForecaster(**params)
    hmodel.fit(train.loc[hm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[hm])
    hpreds = hmodel.predict(train.loc[hv, ['Store', 'Dept', 'Date', 'IsHoliday']])
    wmae_holdout = wmae(y_train[hv].values, hpreds, train.loc[hv, 'IsHoliday'].values)

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    mlflow.log_metric('wmae_holdout', wmae_holdout)
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores)),
               'wmae_holdout': wmae_holdout})
    wandb.finish()

fold_scores, wmae_holdout

In [ ]:
with mlflow.start_run(run_name='TimesFM_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='TimesFM_Final', config=params, reinit='finish_previous')

    pipeline = Pipeline([
        ('model', TimesFMForecaster(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

In [ ]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_TimesFM.ipynb in Colab"
    !git push